# How Much Does Reconstructor Size Matter?

`Tutorials/Advanced/CNNArchitectureComparison.ipynb` compares six *different* CNN architectures at a fixed, matched parameter budget (~0.9M), so that comparison isolates architecture from capacity. This notebook asks the complementary question: holding the architecture fixed (`ClassicCNN`, the plain non-grouped conv baseline from that notebook), how much does the reconstructor's *size* alone matter? Four sizes are trained under otherwise identical conditions -- same instrument, same loss, same training budget, same `Trainer` procedure as `Tutorials/basics/05_TrainingAReconstructor.ipynb` -- spanning roughly 100k to 3M trainable parameters, and compared on training-loss curves and seeded closed-loop steady-state residual wavefront error (nm RMS).

Same instrument setup and reasoning as `CNNArchitectureComparison.ipynb` (see `Ideas/07-cnn-architecture-comparison.md` for the fuller motivation) -- a fictional demo modulated Pyramid WFS + DM, no real bench. `CNNSizeComparison_params.py` is its own params file (not shared with `CNNArchitectureComparison_params.py`) so the two notebooks stay independently editable.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import os

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss, Physics_loss

device = 'cuda'  # set to "cpu" if CUDA is not available


## Configuration and instrument

Identical construction to `CNNArchitectureComparison.ipynb`: a fresh, frozen (`.eval()`) Pyramid WFS + DM, one shared `PhaseDataset`/`FramePreprocess`/loss, reused by every trained size below.

In [ ]:
paramfile = 'CNNSizeComparison_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

PATH = "../../Data/CNNSizeComparison/"
os.makedirs(PATH, exist_ok=True)

wfs = PyramidWFS(WFSParams, device)
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

M2C = dm.MakeZernikeM2C()
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

loss = LogResidualVarianceLoss(dataset.pupil, wfs.wavelength) + Physics_loss(wfs=wfs)

n_channels = wfs.pupil_centers.shape[0]
Nout = framePreprocessor.Nout
Nmodes = DMParams["Nmodes"]
print(f"Pyramid pupils: {n_channels}, preprocessed pupil size: {Nout}x{Nout}, Nmodes: {Nmodes}")

## `ClassicCNN`, at four sizes

`num_downsamples`/`conv_stage`/`ClassicCNN` are copied verbatim from `CNNArchitectureComparison.ipynb`: `num_downsamples` picks the number of VGG-style stages from the pupil crop's actual resolution (`Nout=46` gives 4 stages), and `base_channels` sets the width of the first stage (doubling every stage after). Only `base_channels` varies across the four sizes compared below -- `TrainParams['BaseChannelsList']` (`[10, 18, 30, 50]`, chosen by an actual parameter-count sweep, not hand arithmetic) gives roughly `120k, 380k, 1.05M, 2.9M` trainable parameters, log-spanning the requested ~100k-3M range in four steps.

In [ ]:
def num_downsamples(size, min_size=2):
    """How many stride-2 poolings a feature map of this spatial size can
    take before dropping below min_size pixels -- lets the network downsample
    as deep as the actual pupil-crop resolution allows."""
    n = 0
    while size // 2 >= min_size:
        size //= 2
        n += 1
    return n


def conv_stage(in_channels, out_channels, activation=nn.LeakyReLU):
    """Two 3x3 convs at out_channels, then a 2x downsampling MaxPool --
    one VGG-style stage."""
    return [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        activation(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        activation(),
        nn.MaxPool2d(2),
    ]


class ClassicCNN(nn.Module):
    """Plain, non-grouped Conv2d stack -- channels are just channels, no
    per-pupil structure and no positional information. Copied verbatim from
    CNNArchitectureComparison.ipynb; base_channels is the only knob varied
    across the four sizes compared in this notebook."""

    def __init__(self, n_channels, Nmodes, Nout, base_channels):
        super().__init__()

        n_stages = num_downsamples(Nout)
        channels = [n_channels] + [base_channels * 2 ** i for i in range(n_stages)]

        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1])
        layers.append(nn.AdaptiveAvgPool2d(1))

        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        x = self.encoder(x)
        return self.head(x)


## Instantiating all four sizes

Each size is built once here and printed with its total trainable parameter count and its `base_channels`.

In [ ]:
sizes = {}
for base_channels in TrainParams['BaseChannelsList']:
    model = ClassicCNN(n_channels, Nmodes, Nout, base_channels).to(device=device)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    name = f"{total_params / 1e3:.0f}k (base={base_channels})"
    sizes[name] = model
    print(f"{name:22s} -- total trainable parameters: {total_params:,}")


## Training

Following `05_TrainingAReconstructor.ipynb`'s exact procedure, repeated once per size: a fresh `AdamW` optimizer and a fresh `Trainer` per size, all sharing the same `wfs`/`dm`/`framePreprocessor`/`dataset`/`loss` objects, each calling `trainer.train(TrainRunNb, ClosedLoopIterations)` for the **same** number of steps regardless of size -- the question this notebook asks is what a fixed training budget buys as capacity grows, not whether a bigger network can eventually be trained to convergence given unlimited steps.

Same trade-off as `CNNArchitectureComparison.ipynb`: `Trainer.train()` redraws a fresh, independent atmosphere at every outer step (`PhaseDataset.__getitem__`'s `idx == 0` branch), so training the four sizes one after another on the same shared `dataset` object does not give them the identical per-step realization a hand-rolled shared-draw loop would. Fine for training itself, but it means the training-loss curves below are directionally informative rather than a rigorously paired test; the seeded closed-loop rollout further down is the true apples-to-apples comparison.

In [ ]:
TrainParams['TrainRunNb'] = 5000
TrainParams['ClosedLoopIterations'] = 1

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, model in sizes.items():
    print(f"=== Training {name} ===")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=True)

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )

    checkpoint_name = name.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
    try:
        trainer.load_checkpoint(PATH + f"{checkpoint_name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by a
        # different base_channels/BaseChannelsList choice.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'])

    trainer.save_checkpoint(PATH + f"{checkpoint_name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal


## Comparing training-loss curves

Same overlay approach as `CNNArchitectureComparison.ipynb`: `Trainer.plot_losses` only handles one tracker pair, so all four are overlaid manually here, along with a single oracle "ideal loss" reference curve (a property of the atmosphere/noise/DM, not of network size).

In [ ]:
def smooth(x, window=100):
    x = x.detach().cpu().numpy()
    return np.convolve(x, np.ones(window) / window, "valid")

fig, ax = plt.subplots(figsize=(8, 5))
for name, tracker in loss_trackers.items():
    ax.plot(smooth(tracker), label=name)

first_ideal = next(iter(loss_trackers_ideal.values()))
ax.plot(smooth(first_ideal), label="Oracle bound (ideal)", color="black", linestyle="--")

ax.set_xlabel("Iteration")
ax.set_ylabel(r"Loss")
ax.legend()
plt.show()


## Closed-loop residual wavefront error vs. parameter count

Same seeded-rollout protocol as `CNNArchitectureComparison.ipynb`: reseed immediately before each size's rollout so all four see identical wind/r0/noise draws, then compute the steady-state residual wavefront error in nm RMS (excluding the leaky-integrator warm-up). Plotted against parameter count on a log-x axis -- the actual scaling curve this notebook exists to produce: does closed-loop performance keep improving with capacity under this fixed training budget, or flatten out (or get worse, if the larger sizes are undertrained relative to their capacity at this same `TrainRunNb`)?

In [ ]:
def residual_nm_rms(result, pupil, warmup_fraction=0.3):
    """Steady-state residual wavefront error (nm RMS) over the pupil, from an
    EvaluationResult's residual_opd trajectory, excluding the leaky-integrator
    warm-up period (same 0.3 convention Trainer.evaluate() uses internally)."""
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state = result.residual_opd[warmup:]
    pupil_values = steady_state[..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9


eval_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset.generateClosedLoop = True

nm_rms = {}
param_counts = {}
for name, trainer in trainers.items():
    torch.manual_seed(TrainParams['Seed'])
    result = trainer.evaluate(n_steps=TrainParams['TestRunNb'], dataset=eval_dataset)
    nm_rms[name] = residual_nm_rms(result, dataset.pupil)
    param_counts[name] = sum(p.numel() for p in sizes[name].parameters() if p.requires_grad)
    print(f"{name:22s} -- steady-state residual: {nm_rms[name]:.1f} nm RMS")


In [ ]:
names = list(nm_rms.keys())
xs = [param_counts[n] for n in names]
ys = [nm_rms[n] for n in names]

order = np.argsort(xs)
xs_sorted = [xs[i] for i in order]
ys_sorted = [ys[i] for i in order]
names_sorted = [names[i] for i in order]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(xs_sorted, ys_sorted, "o-")
for x, y, name in zip(xs_sorted, ys_sorted, names_sorted):
    ax.annotate(name, (x, y), textcoords="offset points", xytext=(0, 8), ha="center")

ax.set_xscale("log")
ax.set_xlabel("Trainable parameters")
ax.set_ylabel("Steady-state residual (nm RMS)")
ax.set_title(f"ClassicCNN: closed-loop performance vs. size (fixed {TrainParams['TrainRunNb']}-step training budget)")
plt.tight_layout()
plt.show()


## Visualizing the best size's closed loop

A closer look at whichever size came out on top above -- the same rollout-animation pattern `05_TrainingAReconstructor.ipynb`/`CNNArchitectureComparison.ipynb` use.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

best_name = min(nm_rms, key=nm_rms.get)
print(f"Best size: {best_name} ({nm_rms[best_name]:.1f} nm RMS)")

n_frames = 60
torch.manual_seed(TrainParams['Seed'])
result = trainers[best_name].evaluate(n_steps=n_frames, dataset=eval_dataset)

fig, axes = imshow_multiple(
    [
        {"tensor": result.opd[0], "title": "Input OPD", "same_scale": True},
        {"tensor": result.residual_opd[0], "title": "Residual OPD", "scale_reference": result.opd[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
    ],
    max_channel_number=9,
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.opd[i], "title": "Input OPD", "same_scale": True},
            {"tensor": result.residual_opd[i], "title": "Residual OPD", "scale_reference": result.opd[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
        ],
        fig=fig, axes=axes,
        max_channel_number=9,
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())
